# 03a · Risk #3: where the TRID labels go (2D GRE)

Segment boundaries come only from `TRID` labels. A segment runs from a block
carrying a TRID up to, not including, the next TRID. Two rules follow:

- **Strict:** every instance of a segment has the same blocks, with the same
  kinds of events (RF, gradients, ADC, trigger) in each. Amplitudes, phases,
  frequencies and the length of pure delays may change.
- **Soft:** keep segments short, about one TR. The whole segment is loaded
  into the waveform memory of the sequencer, which is limited.

This notebook breaks each rule with the gre2d.

Run the next cell once per session. On Colab it installs the course package
and GNU Octave (about 2 minutes); on your own computer, follow the README
instead and skip it.

In [ ]:
!pip -q install git+https://github.com/GEHC-External/pulseq-ge-course && python -m pgecourse.colab

In [ ]:
import numpy as np
import pypulseq as pp

import pgecourse

ge = pgecourse.GE()  # downloads the Pulseq, pulseg and pge2 toolboxes on first use

In [ ]:
system = pp.Opts(
    max_grad=50 / np.sqrt(3),
    grad_unit="mT/m",
    max_slew=120 / np.sqrt(3),
    slew_unit="T/m/s",
    rf_dead_time=0,
    rf_ringdown_time=0,
    adc_dead_time=0,
    rf_raster_time=2e-6,
    adc_raster_time=2e-6,
    grad_raster_time=4e-6,
    block_duration_raster=4e-6,
    B0=3.0,
)

Events as in notebook 00, and the three TRID labels used below.

In [ ]:
fov = 240e-3
n = 128
slice_thickness = 5e-3
dwell = 20e-6
tr_delay = 5e-3
n_dummy = 20
pislquant = 10

rf, gz, _ = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(6),
    duration=3e-3,
    slice_thickness=slice_thickness,
    apodization=0.42,
    time_bw_product=4,
    system=system,
    return_gz=True,
    use="excitation",
)
gz_reph = pp.make_trapezoid("z", area=-gz.area / 2, duration=1e-3, system=system)
delta_k = 1 / fov
gx = pp.make_trapezoid("x", flat_area=n * delta_k, flat_time=n * dwell, system=system)
adc = pp.make_adc(n, duration=gx.flat_time, delay=gx.rise_time, system=system)
gx_pre = pp.make_trapezoid("x", area=-gx.area / 2, duration=1e-3, system=system)
pe_areas = (np.arange(n) - n / 2) * delta_k
gy_pre = pp.make_trapezoid("y", area=np.max(np.abs(pe_areas)), duration=1e-3, system=system)
pe_scales = pe_areas / gy_pre.area
gx_spoil = pp.make_trapezoid("x", area=2 * n * delta_k, system=system)
gz_spoil = pp.make_trapezoid("z", area=4 / slice_thickness, system=system)

TRID = [pp.make_label(type="SET", label="TRID", value=v) for v in (1, 2, 3)]

## Case A: dummy TRs share the imaging TRID

It is tempting to label every TR the same way. The dummy TRs have no ADC,
the others do.

In [ ]:
seq_a = pp.Sequence(system)
rf_phase = rf_increment = 0.0
for iy in range(-n_dummy - pislquant, n):
    rf.phase_offset = adc.phase_offset = np.deg2rad(rf_phase)
    rf_increment = (rf_increment + 117) % 360
    rf_phase = (rf_phase + rf_increment) % 360
    pe = pe_scales[iy] if iy >= 0 else 0.0

    seq_a.add_block(rf, gz, TRID[0])  # same TRID for every TR
    seq_a.add_block(gx_pre, pp.scale_grad(gy_pre, pe), gz_reph)
    if iy < -pislquant:
        seq_a.add_block(gx)  # dummy TR: no ADC
    else:
        seq_a.add_block(gx, adc)
    seq_a.add_block(gx_spoil, pp.scale_grad(gy_pre, -pe), gz_spoil)
    seq_a.add_block(pp.make_delay(tr_delay))

try:
    ge.from_seq(seq_a)
except pgecourse.PGE2Error as err:
    print("pulseg.fromSeq:", err)

The first instance of TRID 1 is a dummy TR, so the segment definition has a
readout block *without* ADC. When a later instance asks for an ADC in that
block, the definition has none to play. pulseg stops here. A sequencer that
accepted it would play the scan without acquiring a single sample.

The same applies to any event that is present in some instances and absent
in others: an RF pulse turned off, a gradient present only on some lines,
a block inserted with an `if`. To vary timing, keep a pure delay block in
every instance and change only its length.

## Case B: one TRID for the whole scan

Label only the very first block:

In [ ]:
seq_b = pp.Sequence(system)
rf_phase = rf_increment = 0.0
for iy in range(-n_dummy - pislquant, n):
    rf.phase_offset = adc.phase_offset = np.deg2rad(rf_phase)
    rf_increment = (rf_increment + 117) % 360
    rf_phase = (rf_phase + rf_increment) % 360
    pe = pe_scales[iy] if iy >= 0 else 0.0

    if iy == -n_dummy - pislquant:
        seq_b.add_block(rf, gz, TRID[0])  # first TR only
    else:
        seq_b.add_block(rf, gz)
    seq_b.add_block(gx_pre, pp.scale_grad(gy_pre, pe), gz_reph)
    if iy < -pislquant:
        seq_b.add_block(gx)
    else:
        seq_b.add_block(gx, adc)
    seq_b.add_block(gx_spoil, pp.scale_grad(gy_pre, -pe), gz_spoil)
    seq_b.add_block(pp.make_delay(tr_delay))

print(ge.from_seq(seq_b))
print(ge.check())
print("pge2.validate:", ge.validate())

Every check passes. But the scan is now one segment of 790 blocks, played
once, and pge2 has to load all of it into the sequencer's waveform memory at
the same time. Its size grows with the number of TRs, while the memory
does not, so a longer version of the same scan, or the same scan on another
system, may fail to load.

The opposite extreme, a TRID on every block, is not better either. Each
segment ends with a 117 µs gap on the scanner, gradients have to return to
zero at every segment boundary, and the code no longer says what a TR is.

## Your turn

Label the TRs so that there are three segments, `dummy`, `receive_gain`
and `acquire`, each one TR long. Check that:

- `pulseg.fromSeq` reports 3 segments with 20, 10 and 128 instances;
- `pge2.check` and `pge2.validate` pass.

In [ ]:
seq = pp.Sequence(system)
rf_phase = rf_increment = 0.0
for iy in range(-n_dummy - pislquant, n):
    rf.phase_offset = adc.phase_offset = np.deg2rad(rf_phase)
    rf_increment = (rf_increment + 117) % 360
    rf_phase = (rf_phase + rf_increment) % 360
    pe = pe_scales[iy] if iy >= 0 else 0.0

    trid = ...  # TODO: TRID[0], TRID[1] or TRID[2], depending on iy
    seq.add_block(rf, gz, trid)
    seq.add_block(gx_pre, pp.scale_grad(gy_pre, pe), gz_reph)
    if iy < -pislquant:
        seq.add_block(gx)
    else:
        seq.add_block(gx, adc)
    seq.add_block(gx_spoil, pp.scale_grad(gy_pre, -pe), gz_spoil)
    seq.add_block(pp.make_delay(tr_delay))

print(ge.from_seq(seq))
print(ge.check())
print("pge2.validate:", ge.validate())

In [ ]:
# @title Solution (double-click to show the code)
seq = pp.Sequence(system)
rf_phase = rf_increment = 0.0
for iy in range(-n_dummy - pislquant, n):
    rf.phase_offset = adc.phase_offset = np.deg2rad(rf_phase)
    rf_increment = (rf_increment + 117) % 360
    rf_phase = (rf_phase + rf_increment) % 360
    pe = pe_scales[iy] if iy >= 0 else 0.0

    if iy < -pislquant:
        trid = TRID[0]  # dummy
    elif iy < 0:
        trid = TRID[1]  # receive gain
    else:
        trid = TRID[2]  # acquire
    seq.add_block(rf, gz, trid)
    seq.add_block(gx_pre, pp.scale_grad(gy_pre, pe), gz_reph)
    if iy < -pislquant:
        seq.add_block(gx)
    else:
        seq.add_block(gx, adc)
    seq.add_block(gx_spoil, pp.scale_grad(gy_pre, -pe), gz_spoil)
    seq.add_block(pp.make_delay(tr_delay))

print(ge.from_seq(seq))
print(ge.check())
print("pge2.validate:", ge.validate())

## Take-home

- Put a TRID on the first block of every TR, and use one TRID value per
  *kind* of TR: dummy, calibration, imaging, navigator...
- Within a TRID, never add or remove a block or an event. Vary amplitudes,
  phases and the length of pure delays instead.
- Next, the same rules on longer sequences: an inversion-prepared 3D
  sequence (03b, MPRAGE) and a multislice sequence with a pure delay
  segment (03c).